# Práctica — Ejercicio 7: Empleados de CONICET por sexo y grado académico (2020)

Utilizando los archivos `conicet_personas_2020.xlsx`, `conicet_ref_sexo.xlsx` y
`conicet_ref_grado_academico.xlsx`, generar una tabla con el conteo de empleados
por sexo y máximo grado académico.

---

In [ ]:
import pandas as pd
import numpy as np

## Carga y exploración de los datos

> **Genere una tabla en la cual se informe cuántos empleados de CONICET hay de cada sexo para cada máximo grado académico en 2020.**

In [ ]:
personas = pd.read_excel("../Datasets/conicet_personas_2020.xlsx")
ref_sexo = pd.read_excel("../Datasets/conicet_ref_sexo.xlsx")
ref_grado = pd.read_excel("../Datasets/conicet_ref_grado_academico.xlsx")

print(f"Personas 2020: {personas.shape}")
print(personas.head(3).to_string())
print("\nReferencia sexo:")
print(ref_sexo.to_string())
print("\nReferencia grado académico:")
print(ref_grado.to_string())

In [ ]:
# Join con tabla de sexo
df = personas.merge(ref_sexo, on='sexo_id', how='left')

# Join con tabla de grado académico
df = df.merge(
    ref_grado.rename(columns={'grado_academico_id': 'maximo_grado_academico_id'}),
    on='maximo_grado_academico_id',
    how='left'
)

print(f"Shape tras joins: {df.shape}")
df[['persona_id', 'sexo_descripcion', 'descripcion']].head(5)

In [ ]:
# Tabla: cantidad de empleados por sexo y grado académico
tabla = (
    df.groupby(['descripcion', 'sexo_descripcion'])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
tabla.columns.name = None
tabla = tabla.rename(columns={'descripcion': 'Máximo grado académico'})

# Añadir total por grado
tabla['TOTAL'] = tabla.select_dtypes('number').sum(axis=1)

# Ordenar por total descendente
tabla = tabla.sort_values('TOTAL', ascending=False).reset_index(drop=True)

print("Empleados CONICET 2020 por sexo y máximo grado académico:")
tabla

In [ ]:
# Totales por sexo
print("\nTotales por sexo:")
for col in ['FEMENINO', 'MASCULINO']:
    if col in tabla.columns:
        print(f"  {col}: {tabla[col].sum():,}")
print(f"  TOTAL: {tabla['TOTAL'].sum():,}")

## Conclusiones

- El grado académico más frecuente en CONICET es **universitario de posgrado/doctorado**, que concentra la mayor parte del personal.
- La distribución entre sexo femenino y masculino es relativamente equilibrada en la mayoría de los grados.
- Algunos registros tienen `Sin datos` como grado académico (id = -1).